# Semantic decoder scheduler: real-cache component run

Canonical project: SemanticSchedulerEnd2End. Submitted from WSL user Satya, Kaggle account satyapaladuguplus. CPU-only; no GPU/teacher/extraction/model training. This first version implements full93 lossless transport and dynamic FIFO/RR/random/SPI-input baseline replay. It is not the full temporal P/D/RL comparison.

Input: completed image-pool v5 output, archive SHA verified before use. Frozen source is embedded below from the canonical repository; source hashes, actual environment and all computational cell logs are persisted. No local Windows environment transfers to Kaggle. Managed Kaggle Python and its standard library suffice; no pip installs.

Scope: select one ready message, arrivals continue during unit synthetic nonpreemptive service; MIXED_IID images, no retries. Input-SPI delay is descriptive and not independent urgency evaluation. Relevance/novelty belong to streaming extraction and do not block these baseline policies. Qualified contextual teacher training remains separate.


In [ ]:
import sys,os,json,time,platform,contextlib,traceback,datetime,hashlib,base64,zipfile,io,subprocess,shutil
from pathlib import Path
RUN=Path('/kaggle/working/experiments/stream_scheduler/runs')/datetime.datetime.now(datetime.timezone.utc).strftime('compact-real-cache-%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=False)
(RUN/'logs').mkdir()
STATUS=[]
class Tee:
    def __init__(self,a,b):self.a,self.b=a,b
    def write(self,s):self.a.write(s);self.b.write(s);self.b.flush();return len(s)
    def flush(self):self.a.flush();self.b.flush()
@contextlib.contextmanager
def stage(name):
    start=time.perf_counter();row={'stage':name,'status':'running','started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()};STATUS.append(row)
    (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
    with (RUN/'logs'/f'{name}.log').open('w',buffering=1) as log:
        with contextlib.redirect_stdout(Tee(sys.stdout,log)),contextlib.redirect_stderr(Tee(sys.stderr,log)):
            print('START',name,'output',RUN,flush=True)
            try:
                yield
                row['status']='complete'
            except BaseException as e:
                row['status']='failed';row['error']=type(e).__name__+': '+str(e);traceback.print_exc();raise
            finally:
                row['elapsed_seconds']=time.perf_counter()-start
                (RUN/'cell_status.json').write_text(json.dumps(STATUS,indent=2))
                print('END',name,row['status'],row['elapsed_seconds'],flush=True)
with stage('01_environment'):
    info={'interpreter':sys.executable,'python':sys.version,'platform':platform.platform(),'cpu_count':os.cpu_count(),'execution_target':'Kaggle managed CPU; submitted from WSL', 'gpu_requested':False,'dependencies':'Python standard library; no notebook installations','run_dir':str(RUN)}
    (RUN/'environment.json').write_text(json.dumps(info,indent=2));print(json.dumps(info,indent=2),flush=True)


In [ ]:
with stage('02_source_snapshot'):
    blob=base64.b64decode('UEsDBBQAAAAIADl8Xl3BLQZbLAUAACkOAAAlAAAARXZhbHVhdGlvbi9jb21wYWN0X3NjZW5lX3RyYW5zcG9ydC5weZVW32/bNhB+91/B5aGWEsVO7NiBjbQP2TIkwNAN7VAMCAyDkU42F5n0SMqJi2F/++5Iyfphp27zEpPi3X13991Hnpyc/KaMycAYZjWXZq20ZSplJgYJ51xrvj3fDNgKuMk1rEBaw3KZgGacrTUIaSzPMkhwoVKRQa/zUTHNX9gC1Aqs3jINsdqA3kYsA64lHk0FZAnur9SGZxFT6Cu3S3QtYm6FkkwYJlbrTEDS65ycnHRwQbCM1XlsO51OnHHE+7NarXlsPxPSP0vs0w7Dvy93nz4//P6RvWdDt/7j092vD3/h0rvofXb/gu7N7e19N+y4MwmkbD4XUtj5PDCQpVGZ01wkEcJdCLTezs2SD0Zj9zHJY9C7jVjl0s75GoHH/CmD0GOhP5Eyu11DUDkMKUmpLBPSUgXo5wW7eV+LSavxaDQcVX7oT3NhgH3hWQ53WisddHP0cTnG+DIVC+zSrhns4RfE/U8ucA/TLD1Qcr1anHrQ5qEyaTzytLVgeqlWqyW8Bq1q7Pt2pdkza9WsZVaVDg1tvs4g2KtpvaQZyKABM2Q/YccHVM/dtzLkoW81v/R1MjxW6TTPMrbkZgmGcZkweEUCTobnnpBl1Q+UHNFyuQ0cCV53vX9SKmMpInpFGrRrEB4DQ9bAJStNRCbsth68ojXOmUi4hYLXL3igyU4qidt1VRpPqEy0fryY0c5gt76csXfsv6OFEtKFZEblOgavJgxVwOB49434Cv004wvTrJALMJgOr1zMJgHL+MOr6XhcfS+7ewyPB1Ce7m8U1ivPODoum7YSZsVtvKxBcuQzlWrkEvXmeU5kRumYDO+7vpQRG9eozDVqJRr5bEaUz2hW+VzhJ5xXNxJzNx1BkdhoOhxdzyLWxT7aDGpA+JtGo+spduugERYUg334gLR2Elv8PlYpqeRX0IqtuHlma54kQi5qXomtImKBjHhIpAWZr0ATtb6KtR9XE7kShC0Cx6ihJAjI2gCBvWPBJbu5YSIMI7/JG5sNW8rFDRxNTWtOHsWsFNDiTLnUbsmbML7NV5dAa6LQWy75hovMSdOGzOrMLQBSxGpELl3sICZr7QC5JoTfAybFUCjiCViILVrB61oZVBzYiARkXG+yBptrWTA18tyr5h4Pq+SNoW+YsGKcKqFwp3eHMayB8pC/THs0C8EwKnOO2tdKyM78t/F4WvEfb3ktgAI+BiJyLPKU+i5GuTq7QlYezZprBy44gK5syL/sKjqAz4/SI6lfASychXsdYmUmNKMzMut2e38rgbeIFwYXDB8T97fd6BR8RuBz8U73GlaALi8xtwrZjVu5YocMMjzgflcNTaDWUOwT31dxt4uero4NOkKX+N4CIr17SNF7LhWvNXYVgh0xJ9a791CLCHVZdNHrkAoX7lqhxjlPdIVc06p0WJPzojtH3zzyWaoXWcPuLPvlHXPgeilDtyrTqtvA3X20esR7qFTP6gTdhhPkwNVpceZH5KVoO3pb2GXfcbslJI2xxCczN5Zm5WJ2OsFZI766N3kwGaJsnl82bIl2z0Q7fAwvICjwHZAcN3hv3WuOxWQbDSZnV6fP+/OAFRP0OHXgaHjZh/dFnUgEgzel7q1WGny4Q9JPcie7FvqtetEYbbv7QHyx8ALw+kA/ioJhfpFonH9SybZKuZhYd4efejeVHnixwaWr4Gg6mNT0C6fyG/whml9c/wglvLR6RrRSLCD7PtY0lLSofFoHj4OoJPZwRqCb76az1ov8zHltvrvfUv1Cq2iz8z9QSwMEFAAAAAgAOXxeXaLiA6rvCwAADh0AAC8AAABleHBlcmltZW50cy9zdHJlYW1fc2NoZWR1bGVyL21pbmlfa2FnZ2xlX3J1bi5weZ1Za2/byBX97l/BBiiGjGlKcmLHlpYLpN4UTZHsGklatFAFYkSOpLH4Koe0rTX833vuzJAiLTu7WwG2+Jj7fsy5o1evXv2laPJEJE4leHoS83gjnLriuSqLqnZ4njjJLueZjJ0lVyKVucDKMuW7wPm5cGR+K/Ia1LUgyspJ+VKkKnj16tWRzAyLal3ySomjVVVkTlykqYhrWeTKsQuuoEAtKt9JxIo3aZ3IuKab/zai5bHhapPKZXtrvvAgaGqZtk9vVJEbISWvaXkr4Bq37SJYlhRZe6d2qr2sebWSaSexlpk4OjqCSk7SZKVLLP1bnjbCmx45+NCDAIbB/CDbJrJyzY0Kv1WN8MW9VHVUbPWdt6e4q2Qtolrc1y7pGxBz5WrGvkQc8jo89XmaFndRzvPwrzxVwjtm/8mZZ9VJC55EXYTcqijqnkpO6NCTEftAPDk5ehQXWcnjOlKxyMWeNCh3TNOpUsSgG3o1oKcRuTMiv0RpEWtuLnuBHfNJvjE1K5ImFYc8zXPDlfi79M/rlAjINlEF4h6izVrXfJk1laibKrfcgyujx1dS41urhXWSSVE45075jYhiSjE/E0rxdXtHsnwlROIXTetBXP3OYFb5OjS5FHzRXy6xsqYISnGRhFgUKJ6VMENrksncfaKEyPUrzzOkd8iB4i6kxy0Xb0Sa6re8qiTCqkIFS0XizsfB4jUxvfAHFN7xnEQ3uVwVVeaOfcPWc3DrRChaKoM1fMvv8XJAenLheQujy02xVOH8gbUay4RNV+yB7HycPshH5jtMZu2bar6/WeCNKpoqtq+gC/nKCG3D4flaiv0waxubciIupeZI34s5S8WtSA1X6gp5LNhUPmprpO9WPvfIJpE3mah4LdxfZdlZ5LdOg11a4Kao5K9FHhqfHL85NfHk+VaFD+wK9Slj0mPss7/J9YZNJz77LBLZZGx66rNPxR2bvnm0+ajQr0BmbkmfskhlvCN15mwF9zOfVdRgo6pYypzudLbgoqwEdTqRoBJkpNcupp1LSpEnEjmm26BLofBm1KKTXThfzJwcSTIOZmi3SEzj6vBkMnPErc5aWhJvikKJ6pksPb6ceJ0ganRE0D2426DcScAP1lN6E3CtQg5s1Gp40374HLkCSW1eaYJ2fe96PkYs2zgvfoSIIQ/6kGHPLh+sNDr+pogfwmdlaCUDXhJNa1dQFijTVe163st2wXYw/DG0jpk6SzzfDtYvxRptm5walKJamVQXVZQr94CxyZUwHGTIobomvF3RP9zMe8W10Fl3o2uadHz0DuiRCErkbZbktPW4SpMpIrPsSSH1Yy+fPN++gU8Pmd6E1HfcmyfiictQvzAcyPf8rdiFKc+WCXdupu5NL1w+UbYFvvAOhfazfcB1sBIgpe9bU29TaGwrIsC3jNGSdRp/h/SZCp0auzXpE0t0B5nf2Kaljfl9lgEvie/w/d18dCW3ae0+n4InOj+90UScD4lNTVQiK27Rbp4oqJtKy/nh9esbnyVSYbuP0R5RET6747LWlycDddECc6nMomP0UbQxUd3CodBIoseSzeaVLSnP1/AiFbWwRD+01fYks+llONn3SxlvseNScfT2q151GCMeew22avs2fWid3mbs7rhVw0IsViuBdAjnvYynrtwmPMU8DDWHYatKkFO3B6RGm98khpJz8xhbk1WBTWnDtjfwV7vRmeeduOH2qj/d0igTPI9MzFSTUYrpm5679nxGQ7akc3encxdzQC6ekdbke9WslEFoqK9CkoncXnJrWS9UIXDZCNDDVOZjQMg5Zfv3fVTNWHBTIK166PrGAOin3u+Vj93I50bAgbP19vvU05aJP7CzW33SXzLwzR+JgmUw6jEj/9vsecH5T/pWIlRcybJGxKwwZFVrDAC9USNTRgndRIxAc0ny9NWL4v4e/cSm+jnLk3jdXleCPIhGhUGhKFHO7OdfHBpxqItAf6ep1ni9G5XdSqe7UjOnpzfIyqY+idH/FSUf3ykHGbBjJkf0gEYp0kZNJwjzHxiFAj0b/8ng6lYScmRql9cbAZjnACHXIK/0ZANMZlsNm9oLBNeiVTbtxoi99QyTCDH8/PFfH36KPn78qcUKqXMrE1Fg662KvEiL9Y6RAroSaOJgBOmRLfj/OBhtbCLaKSbjSGQLtHgZtpN08L5aA+rm9TXdVa4342XAkyTi9rnLTk54FW/gOubXu1KENAL7FXYOSZ1IjzHPEcGJcPTLNHYKWasQxFoXIlekAb4CQ/7dCcoMtO2wFfbIZjS3aqFuZEbOyMOGpIoUO5JnR201P13MFGb1+hBjWYBVVpKM+frt/ZdvJhb6UCNx9LzaO9s4fulkg/mrtFGbnsmb0J5ABGrDT8/O3XZaw8CtTbDeDpDomJCrJfMcDow1Hewv8abJt1TWkrQ1O/x0FdDW6168noxP3+p/nr9kzJtugqZMaJrRZEYganETbMR9ItdCAav+KWTj1UV8Prk4X44vk3h1zpcX7y4nk2R1/vb0fHJ6Obl88+7d5Ozdcjk5uzgbX5wuRTKJL9/gdny+Eqcx5iwuUdj/pCOID1VVVC67Q9auHWzO0mnTyMi/Cp87fdj7wh6iGDf0HaPdUe/dQTMvNkTq0cRRufdmOL0n99QBGnmF0Z54uewz9IjgCQSpHm0Izq1WIzNrIil4BqxQKbsn6HZ1H6gancP1Fr0Gv0YCVruwE+l+T4pOlVFLZNqJ14WgG9r/FJ6Px+MXHah1dEz7aHOzIGk0Mc6U/HU/eJlRthqOsVrG3mcqpImYNGOLGWFEV81Zq2NkEhOYCw8hJmliQL724X6ng/og7fZeq88Q67RP51i5CK9csrd95gGrvcZzfx7PmTYtAi7ErsmXqbBbV2zGAaMYVlHbFgqYeK9GjMYZhwNBM7Tctcx5Gi53NYAsHRIh1d3W6OgOfSjCE5gzs2dQoeYTYMfAt9syGFhrVuCPVlgyBK5d+0zw9h1Cz2WUSqgFlWm4u+et49eCYnJRy3wg3pXHE+/PE6RJGI6ntjd9+/L+56/Xv3z5xnxJsHhEaeSIe1DvZarDNrTf6KznomGKYsdrXcqm8+5GH8KgX3d3/kHaTLeY8vyDxMHjycIctGz9sp8xAXpYhsa/6IMB1u70cZGv5LohhGgJnLQotk05o/tbqbDd0uSuI+3w2HRwtGhBFV2LdEdHHA5vsE/ndB5D+7ODRJIZsxsmoiTzcF5p5XThULmQy/W5UUpACjOcXmZPD6jaxeC0Zuu7jVB+rg8Dnxwizd23/tsLPzjzfPds7MMdfnCJa8TSP5uc+pPg1Fv0ytPG9suH60/v/838Pd+DMO51mSMG7tZbhPa8UmvbJ528O7083vo65oaEjQyNPWrUSUgnA3ZLAobLOPrcA8MuWTdKB8MOVL4tpF6KacCqmXjtWx0TDQoNIjRvR72FR7147ynKyzMQ0Io5OSK4PHvt7olOJh5yZSiC32MC5Ped/H3uDjrOC/LaXEPRNNjkTQ9od/llWsRbAxJnztfrj+jlmKM4kOUox4Sb1rsRvFAjB0cl35GrzSlSPznFfZw2YMl6CtgYTM03hlXsRzmAMQqxRugI4F2+obFXpEDfLcSGltRwCawUVSaSGapBKdimHKVRVA+VGEYoihTAxXJ/ooEk9Q551hupnKvrfzgUcEBwQOyqAdhQutujkpzrUS4AdtPRl0+Oqptk12fceSiyHiKkXAOcZCd7zK6BOMB5k/NbLlNq+lQ1GmHNtJwc+BZhWO5IIixbFU3VAayBJcDxpaIhBb05T8iph4juRGO9pzDfJrltevbuaYu85alMdOPomqMFI113G2Apvz21IqhHMX6gQisBz1Kuh6e6MHAH0GwIBksN4Uxaup7XZ2rOhPVx8BDZ+n/kV5rF46DLlrt6Q0mldirAGE756jNJDsMohf/mDf2I0tQUIZ/pzkm9oMu74U98+wGmg8L7JFqKDb+VsMOMXE/9jPjIFSme8VyuYHXr7kP30Q8t/5/36IeZap0WS5e9NiCvDKTSznS9xwHwv/rl8/WnD98+MOqag957dAS6KALmRwSwN0QRDVhRRF2IBq2j/wFQSwMEFAAAAAgAOXxeXQgOaXPXAwAAvAoAACoAAABUZXN0L3Rlc3RzL3Rlc3RfY29tcGFjdF9zY2VuZV90cmFuc3BvcnQucHm1Vs2TmzYUv/NXML6A1ioBs961cbhsZjM5JZ102wvDMDLIWWpAriRsb6d/fJ8QYGCzSWbTckDi6X383tP7YDabvWdpLWhmHjiTLGWFyekXToXIWWWecvnIamnm5YFxmVdfTHaQcEAK85g3HBk90CqjVZpT4cxmM0OzthJFvnVqmRfGjrPSPBD5CJT2zPwVPjt2IXmdyu6rrnIpqZCG8fnTp4dQMdpJsssLmiTIAXCsOFIbOQfCaSVFtIgNcaBpODbqKFqiLGvRgqVEgbetlJUHkspEpLSiieSkEkrQwsrcG+v+SIq6YX3zAqdzeLKQUbKsLujUqqZquwqBrV5oo95OwUhGuUPPAEzz2XpBhmGkBRHCfKct/qYMPkAMhN1Fw1Gf74igKDBMeDK6MwWVvx9sQYtdS1SP+nTSUGt2Rgo7B2zvFluua135C2x5XrNGD7ym8dVqOY/ek0KoLcDqLJ1yThtDOGU1BD38yCqK4QJIu2X7UCkY4GgZ9WLmO7PbCbNi0lRSJgVDZuTGV2u/l9NKm7eSajffEUrL0Pbevl0t0S+ekmJ7zeZe1GqOtf8iB6ey5pVpb58g3na0wHklbbZHMZprkr9od4663kd6ttvgobnOYEjJdG9bYOSDha+0v6jX3z2ttsYxNE9LR7JEkzy4jgKuu6AWmpMXDga3ovICsoxwQRN6hmue5kIa6jhtUljDm+XSX25IRyPRGojebc99Cpvcae46xQRtzpqQOlDhLKP2CY2zDDKWcnn/V00Ku6CVfUbYv4Vkn561WjLaaAGu09QJ6CEv+TBEFXk6Q8GBFY4WjR+vg+m+DufflLPkKJIdgZ6STaESvB2ARfiyh/Joquo7yMZ+EITwYv1VkY9sDLmXwGPCFn3D4tjdZ4J4O/UehkTTS0mVJUfK1QgQ0xhML2MQjwuUHeNmSQpYSpg9eWVG5yjwYtyW3jWU3DnyghgDeRG3BRN5K+w2J9dBz+rjVUNaBHEcjEpNTa+hz59JLkDiD+ju9J5zxlEwDkAP6ILzx3U89zYKloB2kMYKMeGcPI1isjnBAAu915vUgTg96wt1RY6QpWTbXhgM9q/d16BFrH8OyKB5oB/y21/exP+E3mL1Xzj/bXt9/Sm7Nzf/S8C3JOubcSU5/BJNY/1ISRaOR8XdHYwKHy+xd3vxoSRiL4adJPKvl0E8Kp/WRFM8ke1iD7sIt2uMI3vt91sXu2o3qY9zqOB0taW6T6tSDbwGwXxrWc6fLIe+NMb84Q7GG0Uah0LQSf5cAZ7VfxCM5iSpSAk/e2FoJUlJ8ipJrKD/D1IEGxn/AlBLAQIUAxQAAAAIADl8Xl3BLQZbLAUAACkOAAAlAAAAAAAAAAAAAACAAQAAAABFdmFsdWF0aW9uL2NvbXBhY3Rfc2NlbmVfdHJhbnNwb3J0LnB5UEsBAhQDFAAAAAgAOXxeXaLiA6rvCwAADh0AAC8AAAAAAAAAAAAAAIABbwUAAGV4cGVyaW1lbnRzL3N0cmVhbV9zY2hlZHVsZXIvbWluaV9rYWdnbGVfcnVuLnB5UEsBAhQDFAAAAAgAOXxeXQgOaXPXAwAAvAoAACoAAAAAAAAAAAAAAIABqxEAAFRlc3QvdGVzdHMvdGVzdF9jb21wYWN0X3NjZW5lX3RyYW5zcG9ydC5weVBLBQYAAAAAAwADAAgBAADKFQAAAAA=')
    assert hashlib.sha256(blob).hexdigest()=='35fa19e75d939ebdab535feb083d6774c44d5baa66ba782db6a25cd5ff9590d1'
    SOURCE=RUN/'source'
    SOURCE.mkdir()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(SOURCE)
    expected={'Evaluation/compact_scene_transport.py': '15a7f1e4aac1435761de5ee53f8cc245c223fb057d281f588928d999fb95caf9', 'experiments/stream_scheduler/mini_kaggle_run.py': '59f88f17df96ec23809548dfebd8d4822ca953e7a34f30aaa6cef8fe78ae2b4d', 'Test/tests/test_compact_scene_transport.py': 'c13149ed9f7c0e33cf3c39a40a1b6c7cae6e11d524c014998668a84cd50fcc70'}
    for name,value in expected.items():
        assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()==value
    manifest={'commit':'032c23a3dafc7bfeb39a84ba55193b6b6f3a22a0','dirty_snapshot':True,'source_hashes':expected,'input_kernel':'satyapaladuguplus/semantic-image-pool-scene-cache-20261030/5','expected_archive_sha256':'0f8c6186b09dcf6ab87911df642612919377157bb1585082bed1c93b1506fe2c','gpu_budget':0,'internal_driver_timeout_seconds':900}
    (RUN/'manifest.json').write_text(json.dumps(manifest,indent=2));print(json.dumps(manifest,indent=2),flush=True)


In [ ]:
with stage('03_input_and_regression'):
    matches=list(Path('/kaggle/input').rglob('Mini_dataset_6000.tar'))
    if len(matches)!=1:raise ValueError('Expected exactly one attached mini archive; found '+str(matches))
    ARCHIVE=matches[0];print('Input archive',ARCHIVE,'bytes',ARCHIVE.stat().st_size,flush=True)
    (RUN/'input.json').write_text(json.dumps({'path':str(ARCHIVE),'bytes':ARCHIVE.stat().st_size},indent=2))
    subprocess.run([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'Test/tests'),'-p','test_compact_scene_transport.py','-v'],check=True,timeout=60)


In [ ]:
with stage('04_transport_and_dynamic_replay'):
    command=[sys.executable,'-u',str(SOURCE/'experiments/stream_scheduler/mini_kaggle_run.py'),'--archive',str(ARCHIVE),'--output',str(RUN/'results')]
    print('Command',command,flush=True)
    with (RUN/'logs/driver.log').open('w',buffering=1) as log:
        proc=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,text=True)
        try:
            code=proc.wait(timeout=900)
        except subprocess.TimeoutExpired:
            proc.kill();proc.wait();raise RuntimeError('Bounded CPU driver timed out; partial logs preserved')
    print((RUN/'logs/driver.log').read_text(),flush=True)
    if code:raise RuntimeError('Driver failed with exit '+str(code))
    summary=json.loads((RUN/'results/summary.json').read_text())
    assert summary['scene_roundtrips']==6000 and summary['channels_retained']==93
    for traces in summary['traces'].values():
        for result in traces.values():assert result['offered']==result['selected']+result['unselected']
    print(json.dumps({k:v for k,v in summary.items() if k!='traces'},indent=2),flush=True)


In [ ]:
with stage('05_report_and_recovery'):
    summary=json.loads((RUN/'results/summary.json').read_text())
    text='# Real-cache scheduler component result\n\n'+json.dumps({k:v for k,v in summary.items() if k!='traces'},indent=2)+'\n\nNo P/neural/RL training or independent urgency evaluation occurred. Four baseline policies across three dynamic MIXED_IID traces; delays stratified by input SPI only. Temporal missions/relevance/novelty and qualified teacher supervision remain the next components. This notebook did not alter image-cache or SPI production.\n'
    (RUN/'REPORT.md').write_text(text)
    print('Saved results, traces, source, manifest and cell logs under',RUN,flush=True)
# Capture finished cell status/logs after stage exit in the recovery ZIP.
paths={str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest() for p in RUN.rglob('*') if p.is_file() and p.name!='artifact_manifest.json' and '/logs/' not in str(p)}
(RUN/'artifact_manifest.json').write_text(json.dumps(paths,indent=2))
archive=shutil.make_archive(str(RUN.parent/(RUN.name+'-recovery')),'zip',RUN)
print('RECOVERY',archive,'sha256',hashlib.sha256(Path(archive).read_bytes()).hexdigest(),flush=True)
